# TAM Research — isolated RLT Colab bridge

Runs only the `exp/rlt-colab` lane. The worker is one-shot: it prints an immediate heartbeat, processes the currently runnable bounded jobs, writes create-once results to the same branch, and exits. It never writes to `main` or historical research lanes.


In [ ]:
import torch
print('torch:', torch.__version__, flush=True)
print('cuda:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before starting the bridge.')
print('gpu:', torch.cuda.get_device_name(0), flush=True)


In [ ]:
from google.colab import userdata
from pathlib import Path
import json, os, shutil, stat, subprocess, sys, urllib.parse, urllib.request

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Missing Colab secret GITHUB_TOKEN')
repo_dir = Path('/content/tam-rlt-bridge')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
askpass = Path('/content/rlt-git-askpass.sh')
askpass.write_text('''#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n''')
askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
env = os.environ.copy()
env['GITHUB_TOKEN'] = token
env['GIT_ASKPASS'] = str(askpass)
env['GIT_ASKPASS_REQUIRE'] = 'force'
env['GIT_TERMINAL_PROMPT'] = '0'
env['PYTHONUNBUFFERED'] = '1'
subprocess.run(['git','clone','--depth','1','--single-branch','--branch','exp/rlt-colab','https://github.com/vinceackermann2-sys/tam-research.git',str(repo_dir)], check=True, env=env)
askpass.unlink(missing_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo_dir)], check=True, env=env)
head = subprocess.check_output(['git','-C',str(repo_dir),'rev-parse','HEAD'], text=True).strip()
print('bridge code pinned to:', head, flush=True)
query = urllib.parse.urlencode({'ref':'exp/rlt-colab'})
url = f'https://api.github.com/repos/vinceackermann2-sys/tam-research/contents/experiments/rlt/bridge/jobs?{query}'
req = urllib.request.Request(url, headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28','User-Agent':'tam-rlt-colab-preflight'})
with urllib.request.urlopen(req, timeout=30) as response:
    rows = json.loads(response.read().decode())
job_names = sorted(row['name'] for row in rows if row.get('name','').endswith('.json'))
print('GitHub queue reachable:', len(job_names), 'json job(s)', flush=True)
print('queued:', ', '.join(job_names), flush=True)


In [ ]:
worker = repo_dir / 'experiments' / 'rlt' / 'bridge_worker.py'
print('=== RLT BRIDGE LAUNCH ===', flush=True)
print('starting one-shot worker; do not interrupt this cell', flush=True)
cmd = [sys.executable,'-u',str(worker),'--branch','exp/rlt-colab','--poll-seconds','20','--once']
proc = subprocess.Popen(cmd, cwd=str(repo_dir), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
print(f'=== RLT BRIDGE EXIT rc={rc} ===', flush=True)
if rc != 0:
    raise RuntimeError(f'bridge worker exited with code {rc}')
